# Research · Dehazing — Where should learning enter a physics-based pipeline? (NB 4)

**Group 01 · Dept. of CSE, East West University** — Md. Asif Hossain (2022-3-60-007) · Nabil Subhan (2022-3-60-063) · K M Nudar (2022-3-60-234)

**Course:** CSE 348 / 438 — Digital Image Processing · **Research project** (Idea Bank: Filtering & Restoration → *Image dehazing using the dark channel prior*)

**Paper 2 (AI).** The Dark Channel Prior pipeline is kept as a fixed classical scaffold and **one stage at a time** is
replaced by a small learned module — a residual correction of its classical counterpart — trained with an identical step
budget. Every arm is evaluated on the *same* test split, metrics and paired statistics as NB 2 (paper 1). Plan and
hypotheses: `HYBRID_PLAN.md`.

| Arm | Learned | Training signal |
|---|---|---|
| DCP baseline / **DCP tuned** | — | — |
| **H-A** | atmospheric light (ΔA) | L1 to ground truth *through* the fixed classical stages |
| **H-t (sup)** | transmission refinement (Δt on the guided filter) | L1 to ground truth through the fixed recovery |
| **H-t (prior)** | same network | **no ground truth**: dark-channel-of-output loss + anchor + TV, on hazy images only (incl. real ones) |
| **H-J** | recovery / colour correction (ΔJ) | L1 to ground truth |
| **H-all** | H-A + H-t (sup) + H-J composed | (modules trained separately) |
| **AOD-Net** | everything, end-to-end | L1 to ground truth, same data and budget |

**Attach:** the evaluation datasets of NB 1–3, **NB 2's output** (`dehaze_results.json` → tuned configuration + split
fingerprint), and **RESIDE-ITS** (or OTS) as the training set. Accelerator: *GPU T4* recommended (≈ 1.5–2 h total);
CPU works with `SEEDS=[0]`.


In [ ]:
# ===== Dehazing research library (identical cell in NB1 / NB2 / NB3 — edit in one, copy to all) =====
import os, re, json, time, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
import matplotlib.pyplot as plt
from scipy import sparse, stats
from scipy.sparse.linalg import cg as _cg
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim
from skimage.color import rgb2lab, deltaE_ciede2000
warnings.filterwarnings("ignore")

SEED = 42
IS_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path(os.environ.get("DEHAZE_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("DEHAZE_WORK", "/kaggle/working" if IS_KAGGLE else "dehaze_work")); WORK.mkdir(parents=True, exist_ok=True)
FAST = os.environ.get("DEHAZE_FAST", "0") == "1"          # quick smoke-run: fewer images / bootstraps

CONFIG = dict(
    max_side=512,               # images resized so the longer side <= max_side (metrics computed at this size)
    max_per_dataset=60 if FAST else 500,
    tune_frac=0.30, min_tune=5, # scene-grouped tune/test split (tune never exceeds half the scenes); knobs chosen on TUNE, reported on TEST
    # DCP baseline (He, Sun & Tang 2009/2011 defaults)
    patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3,
    # bright / sky-like region mask (HSV): high value, low saturation
    bright_v=0.75, bright_s=0.25, min_bright_px=200,
    n_boot=300 if FAST else 1000,
    matting_max=6 if FAST else 30, matting_side=320, matting_lambda=1e-4, matting_eps=1e-7,
)
print(f"env: {'Kaggle' if IS_KAGGLE else 'local'} | input {INPUT_ROOT} | work {WORK} | FAST={FAST}")

# ---------- figure style: validated palette (dataviz validator, light surface #fcfcfb) ----------
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_MAGENTA, C_VIOLET, C_RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7", "#e34948"
SERIES = [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_MAGENTA]        # categorical, fixed order (adjacent-pair validated)
RAMP3 = ["#86b6ef", "#2a78d6", "#104281"]                        # ordinal light -> medium -> dense (single-hue, validated)
INK, INK2, MUTED, GRIDC, RULE, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9.5, "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": RULE, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlesize": 10,
    "legend.frameon": False, "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF})
METHOD_COLORS = {"hazy input": MUTED, "CLAHE": C_YELLOW, "DCP baseline": C_BLUE, "DCP tuned": C_VIOLET, "AOD-Net": C_ORANGE}
import matplotlib as _mpl, scipy as _scipy, skimage as _skimage
BOXLBL = "tick_labels" if tuple(int(x) for x in _mpl.__version__.split(".")[:2]) >= (3, 9) else "labels"   # boxplot kwarg renamed in 3.9
print(f"numpy {np.__version__} | opencv {cv2.__version__} | scipy {_scipy.__version__} | scikit-image {_skimage.__version__} | matplotlib {_mpl.__version__} | pandas {pd.__version__}")

def savefig(name):
    """Save PNG (for notebooks/README) and PDF (for the IEEE report) with one call."""
    plt.savefig(WORK / f"{name}.png", bbox_inches="tight"); plt.savefig(WORK / f"{name}.pdf", bbox_inches="tight")

def table_png(df, name, title=None, fmt="{:.3f}", highlight_max=(), highlight_min=()):
    """Render a DataFrame as a publication-style table image (also written as CSV)."""
    df.to_csv(WORK / f"{name}.csv")
    cells = [[(fmt.format(v) if isinstance(v, (float, np.floating)) and not (isinstance(v, float) and math.isnan(v)) else str(v)) for v in row] for row in df.values]
    fig, ax = plt.subplots(figsize=(min(14, 1.15 + 1.35 * len(df.columns)), 0.5 + 0.34 * (len(df) + 1)))
    ax.axis("off")
    tbl = ax.table(cellText=cells, colLabels=list(df.columns), rowLabels=[str(i) for i in df.index], loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 1.35)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(GRIDC); cell.set_linewidth(0.6)
        if r == 0: cell.set_facecolor("#e9eef5"); cell.set_text_props(weight="bold", color=INK)
        elif c == -1: cell.set_text_props(color=INK2)
    for col in highlight_max:
        if col in df.columns:
            j = list(df.columns).index(col); i = int(np.nanargmax(df[col].values.astype(float)))
            tbl[(i + 1, j)].set_facecolor("#dcefe5"); tbl[(i + 1, j)].set_text_props(weight="bold")
    for col in highlight_min:
        if col in df.columns:
            j = list(df.columns).index(col); i = int(np.nanargmin(df[col].values.astype(float)))
            tbl[(i + 1, j)].set_facecolor("#dcefe5"); tbl[(i + 1, j)].set_text_props(weight="bold")
    if title: ax.set_title(title, fontweight="bold", pad=8, color=INK)
    plt.savefig(WORK / f"{name}.png", bbox_inches="tight"); plt.show()

# ---------- Dark Channel Prior: every stage is one named classical operator ----------
def dark_channel(img, patch=15):
    """J_dark(x) = min_c min_{y in Omega(x)} J_c(y): per-pixel channel minimum, then a (patch x patch) grey erosion."""
    mn = img.min(axis=2)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (int(patch), int(patch)))
    return cv2.erode(mn, k)

def estimate_A(img, dark, method="dcp_top"):
    """Atmospheric light A (3-vector in [0,1]).
    brightest     : brightest input pixel (naive; sky/white objects hijack it)
    dcp_top       : He et al. — among the 0.1% brightest dark-channel pixels, the input pixel with highest intensity
    dcp_top_mean  : mean of those 0.1% candidates (robust to single outliers)
    quadtree      : Kim et al. 2013 — recursively keep the quadrant maximising mean - std, then brightest pixel there
    """
    flat = img.reshape(-1, 3)
    if method == "brightest":
        return flat[flat.sum(1).argmax()].copy()
    if method == "quadtree":
        reg = img
        while reg.shape[0] * reg.shape[1] > 400 and min(reg.shape[:2]) >= 4:
            H, W = reg.shape[:2]; hh, ww = H // 2, W // 2
            quads = [reg[:hh, :ww], reg[:hh, ww:], reg[hh:, :ww], reg[hh:, ww:]]
            score = [q.reshape(-1, 3).mean() - q.reshape(-1, 3).std() for q in quads]
            reg = quads[int(np.argmax(score))]
        fr = reg.reshape(-1, 3); return fr[fr.sum(1).argmax()].copy()
    n = max(1, int(0.001 * dark.size))
    idx = np.argpartition(dark.ravel(), -n)[-n:]
    cand = flat[idx]
    if method == "dcp_top_mean":
        return cand.mean(0)
    return cand[cand.sum(1).argmax()].copy()                      # dcp_top

def transmission_raw(img, A, omega=0.95, patch=15):
    """t~(x) = 1 - omega * dark_channel(I / A). omega < 1 keeps a trace of haze for depth perception."""
    return 1.0 - omega * dark_channel(img / np.maximum(A, 1e-6)[None, None, :], patch)

def guided_filter(guide, src, r=40, eps=1e-3):
    """He, Sun & Tang 2010 (grey guide). Local linear model q = a*I + b in (2r+1)^2 windows; O(N) via box filters."""
    k = (2 * int(r) + 1, 2 * int(r) + 1)
    box = lambda x: cv2.boxFilter(x, cv2.CV_64F, k, normalize=True, borderType=cv2.BORDER_REFLECT)
    mI, mp = box(guide), box(src)
    cov = box(guide * src) - mI * mp
    var = box(guide * guide) - mI * mI
    a = cov / (var + eps); b = mp - a * mI
    return box(a) * guide + box(b)

def matting_laplacian(img, eps=1e-7):
    """Levin et al. 2008 closed-form matting Laplacian (3x3 windows) as a sparse matrix — the refinement He et al. 2009 used."""
    from numpy.lib.stride_tricks import sliding_window_view
    h, w, c = img.shape; n = h * w; ws = 9
    idx = np.arange(n).reshape(h, w)
    win_idx = sliding_window_view(idx, (3, 3)).reshape(-1, ws)                                  # (m, 9)
    win_I = sliding_window_view(img, (3, 3), axis=(0, 1)).reshape(-1, c, ws).transpose(0, 2, 1)   # (m, 9, 3)
    mu = win_I.mean(1, keepdims=True); X = win_I - mu
    cov = np.einsum("mki,mkj->mij", X, X) / ws
    inv = np.linalg.inv(cov + (eps / ws) * np.eye(c)[None])
    vals = (1.0 + np.einsum("mki,mij,mlj->mkl", X, inv, X)) / ws                                # (m, 9, 9)
    rows = np.repeat(win_idx, ws, axis=1).ravel(); cols = np.tile(win_idx, (1, ws)).ravel()
    Lw = sparse.coo_matrix((vals.ravel(), (rows, cols)), shape=(n, n)).tocsr()
    return sparse.diags(np.asarray(Lw.sum(1)).ravel()) - Lw

def matting_refine(img, t_raw, lam=1e-4, eps=1e-7, side=320):
    """Solve (L + lam*I) t = lam * t~ at a reduced resolution (conjugate gradients), then upsample."""
    h, w = t_raw.shape; s = min(1.0, side / max(h, w))
    if s < 1.0:
        im = cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
        tr = cv2.resize(t_raw, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_AREA)
    else:
        im, tr = img, t_raw
    L = matting_laplacian(im, eps); n = L.shape[0]
    Aop = L + lam * sparse.identity(n, format="csr")
    try: t, _ = _cg(Aop, lam * tr.ravel(), x0=tr.ravel(), rtol=1e-4, maxiter=800)
    except TypeError: t, _ = _cg(Aop, lam * tr.ravel(), x0=tr.ravel(), tol=1e-4, maxiter=800)
    t = t.reshape(tr.shape)
    if s < 1.0: t = cv2.resize(t, (w, h), interpolation=cv2.INTER_LINEAR)
    return np.clip(t, 0, 1)

def recover(img, A, t, t0=0.10):
    """Invert the haze model I = J t + A (1 - t):  J = (I - A) / max(t, t0) + A. The floor t0 bounds noise amplification."""
    tt = np.clip(t, t0, 1.0)[..., None]
    return np.clip((img - A[None, None, :]) / tt + A[None, None, :], 0.0, 1.0)

def to_gray(img): return cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0

def dehaze(img, patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3, **kw):
    """Full DCP pipeline; returns every intermediate so any stage can be inspected or ablated."""
    dark = dark_channel(img, patch)
    A = estimate_A(img, dark, A_method)
    t_raw = transmission_raw(img, A, omega, patch)
    if refine == "guided":
        t = guided_filter(to_gray(img), t_raw, gf_r, gf_eps)
    elif refine == "matting":
        t = matting_refine(img, t_raw, CONFIG["matting_lambda"], CONFIG["matting_eps"], CONFIG["matting_side"])
    else:
        t = t_raw
    J = recover(img, A, t, t0)
    return {"dehazed": J, "dark": dark, "A": A, "t_raw": t_raw, "t": np.clip(t, 0, 1)}

def clahe_enhance(img, clip=2.0, tile=8):
    """Classical contrast-enhancement control (no haze model): CLAHE on the L channel of Lab."""
    lab = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float64) / 255.0

# ---------- metrics: full-reference (PSNR, SSIM, CIEDE2000) + no-reference (Hautière e, r; DCP density) ----------
def psnr(out, gt): return float(_psnr(gt, out, data_range=1.0))
def ssim(out, gt): return float(_ssim(gt.astype(np.float32), out.astype(np.float32), data_range=1.0, channel_axis=2))
def ciede(out, gt, side=256):
    """Mean CIEDE2000 colour difference in CIELAB; evaluated on an area-downsampled copy (<= side px) — a spatial mean is insensitive to this, 4x faster."""
    h, w = gt.shape[:2]; f = side / max(h, w)
    if f < 1.0:
        gt = cv2.resize(gt, (max(8, int(w * f)), max(8, int(h * f))), interpolation=cv2.INTER_AREA); out = cv2.resize(out, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_AREA)
    return float(deltaE_ciede2000(rgb2lab(gt.astype(np.float32)), rgb2lab(out.astype(np.float32))).mean())

def hautiere(hazy, out):
    """Hautière et al. 2008 blind contrast descriptors: e = rate of new visible edges, r = geometric-mean gradient ratio on visible edges."""
    g0, g1 = to_gray(hazy), to_gray(out)
    e0 = cv2.Canny((g0 * 255).astype(np.uint8), 50, 150) > 0
    e1 = cv2.Canny((g1 * 255).astype(np.uint8), 50, 150) > 0
    n0, n1 = int(e0.sum()), int(e1.sum())
    e = (n1 - n0) / max(n0, 1)
    grad = lambda g: np.hypot(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3))
    G0, G1 = grad(g0), grad(g1)
    m = e1 & (G0 > 1e-3) & (G1 > 1e-3)
    r = float(np.exp(np.mean(np.log(G1[m] / G0[m])))) if m.sum() > 10 else float("nan")
    return e, r

def haze_density(img, patch=15):
    """No-reference haze-density proxy: mean dark channel of the input (DCP's own statistic; 0 = clear, 1 = opaque)."""
    return float(dark_channel(img, patch).mean())

def bright_mask(img, v=None, s=None):
    """Sky-like / bright-region mask (HSV value high, saturation low) — where the dark-channel prior is violated."""
    hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float64)
    return (hsv[..., 2] / 255.0 > (v or CONFIG["bright_v"])) & (hsv[..., 1] / 255.0 < (s or CONFIG["bright_s"]))

def score_all(out, gt, hazy):
    e, r = hautiere(hazy, out)
    return dict(psnr=psnr(out, gt), ssim=ssim(out, gt), ciede=ciede(out, gt), e=e, r=r)

# ---------- statistics: paired bootstrap CI + Wilcoxon signed-rank on the SAME images ----------
def paired_stats(a, b, n_boot=None, seed=0):
    """Paired difference a - b per image: mean, 95% bootstrap CI of the mean, Wilcoxon p."""
    a, b = np.asarray(a, float), np.asarray(b, float); ok = ~(np.isnan(a) | np.isnan(b)); d = (a - b)[ok]
    if len(d) == 0: return dict(mean=np.nan, lo=np.nan, hi=np.nan, p=np.nan, n=0)
    rng = np.random.default_rng(seed); nb = n_boot or CONFIG["n_boot"]
    boots = rng.choice(d, (nb, len(d)), replace=True).mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) and len(d) >= 6 else float("nan")
    return dict(mean=float(d.mean()), lo=float(lo), hi=float(hi), p=p, n=int(len(d)))

def mean_ci(x, n_boot=None, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); nb = n_boot or CONFIG["n_boot"]
    boots = rng.choice(x, (nb, len(x)), replace=True).mean(1)
    return float(x.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ---------- data: discover paired (hazy, clear) benchmarks under /kaggle/input, else synthesise ----------
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
HAZY_DIR = {"hazy", "haze", "hazy_images", "hazy_image", "haze_images", "input", "inputs", "hazed", "synthetic", "hazy_imgs"}
GT_DIR = {"gt", "clear", "clean", "clear_images", "clean_images", "target", "targets", "ground_truth", "groundtruth", "original", "reference", "sharp", "gt_images", "clear_imgs"}

def _dataset_name(path):
    """-> (name, is_real, is_training_set). Training sets (RESIDE ITS/OTS, anything named train) are excluded from evaluation."""
    p = "/" + str(path).lower().replace("\\", "/") + "/"
    tok = lambda *ks: any(re.search(r"[/_\-\s#(](" + k + r")[/_\-\s)]", p) for k in ks)
    if "dense" in p: return "Dense-Haze", True, False
    if tok("nh-haze", "nhhaze", "nh_haze", "nh"): return "NH-HAZE", True, False
    if tok("o-haze", "o-hazy", "ohaze", "o_haze", "o-haz", "ohazy"): return "O-HAZE", True, False
    if tok("i-haze", "i-hazy", "ihaze", "i_haze", "i-haz", "ihazy"): return "I-HAZE", True, False
    if tok("its", "ots", "train", "training"): return ("RESIDE-ITS-train" if tok("its") else "RESIDE-OTS-train" if tok("ots") else "train"), False, True
    if tok("sots", "reside", "nyuhaze500", "nyu"):
        if "indoor" in p or tok("nyuhaze500", "nyu"): return "SOTS-indoor", False, False
        if "outdoor" in p: return "SOTS-outdoor", False, False
        return "RESIDE", False, False
    return Path(path).name or "paired", False, False

def _key(stem):
    k = stem.split("_")[0].split("-")[0]
    d = "".join(ch for ch in k if ch.isdigit())
    return d.lstrip("0") or d or k.lower()

def _images(d): return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)

def discover_pairs(root=INPUT_ROOT):
    """Find every (hazy, clear) pair under root. Handles hazy/ + GT/ sibling folders (RESIDE-SOTS, O/I/Dense/NH-HAZE
    Kaggle mirrors) and flat folders with *_hazy / *_GT suffixes. Pairs by the leading numeric key of the file stem."""
    root = Path(root); recs = []
    if not root.exists(): return recs
    seen = set()
    for d in sorted(p for p in root.rglob("*") if p.is_dir()):
        nm = d.name.lower()
        if nm in HAZY_DIR and _images(d):
            gt = next((s for s in d.parent.iterdir() if s.is_dir() and s.name.lower() in GT_DIR and _images(s)), None)
            if gt is None: continue
            gmap = {}
            for g in _images(gt): gmap.setdefault(_key(g.stem), g)
            name, real, train = _dataset_name(d.parent)
            for hz in _images(d):
                g = gmap.get(_key(hz.stem))
                if g is None: continue
                if (name, hz.name) in seen: continue
                seen.add((name, hz.name))
                recs.append(dict(dataset=name, real=real, train=train, key=f"{name}:{_key(hz.stem)}", name=hz.stem, hazy=str(hz), gt=str(g)))
        else:                                                    # flat layout: 01_hazy.png + 01_GT.png in one folder
            ims = _images(d)
            hz_ims = [p for p in ims if re.search(r"(hazy|haze)", p.stem, re.I)]
            gt_ims = {_key(p.stem): p for p in ims if re.search(r"(gt|clear|clean)", p.stem, re.I) and not re.search(r"(hazy|haze)", p.stem, re.I)}
            if hz_ims and gt_ims:
                name, real, train = _dataset_name(d)
                for hz in hz_ims:
                    g = gt_ims.get(_key(hz.stem))
                    if g is None or (name, hz.name) in seen: continue
                    seen.add((name, hz.name))
                    recs.append(dict(dataset=name, real=real, train=train, key=f"{name}:{_key(hz.stem)}", name=hz.stem, hazy=str(hz), gt=str(g)))
    return recs

def load_rgb(src, max_side=None):
    """Path or array -> float64 RGB in [0,1], longer side <= max_side (aspect preserved)."""
    ms = max_side or CONFIG["max_side"]
    if isinstance(src, np.ndarray):
        im = Image.fromarray((src * 255).astype(np.uint8) if src.dtype != np.uint8 else src)
    else:
        im = Image.open(src); im = im.convert("RGB")
    im = im.convert("RGB")
    if max(im.size) > ms: im.thumbnail((ms, ms), Image.LANCZOS)
    return np.asarray(im, np.float64) / 255.0

def load_pair(rec):
    hz, gt = load_rgb(rec["hazy"]), load_rgb(rec["gt"])
    if hz.shape != gt.shape: gt = cv2.resize(gt, (hz.shape[1], hz.shape[0]), interpolation=cv2.INTER_AREA)
    return hz, gt

# synthetic fallback — keeps every notebook runnable with no dataset attached (results are then labelled 'synthetic')
def _depth_map(h, w, kind, rng):
    yy, xx = np.mgrid[0:h, 0:w]
    if kind == "vertical": d = 1.0 - yy / max(h - 1, 1)                                   # far at the top (sky-like)
    elif kind == "radial": d = np.sqrt(((xx - w / 2) / w) ** 2 + ((yy - h / 2) / h) ** 2) * 1.6
    else:
        d = cv2.GaussianBlur(rng.random((h, w)), (0, 0), max(h, w) / 8); d = (d - d.min()) / (np.ptp(d) + 1e-9)
    return 0.15 + 0.85 * np.clip(d, 0, 1)

def synth_haze(clear, beta, A, depth):
    t = np.exp(-beta * depth)[..., None]
    return np.clip(clear * t + np.asarray(A)[None, None, :] * (1 - t), 0, 1), t[..., 0]

def synthetic_records():
    from skimage import data as skd
    rng = np.random.default_rng(SEED); clears = []
    for nm in ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "retina", "colorwheel", "logo", "camera", "brick", "grass"]:
        try:
            im = getattr(skd, nm)()
            if im.ndim == 2: im = np.stack([im] * 3, -1)
            if im.dtype != np.uint8: im = (255 * (im - im.min()) / (np.ptp(im) + 1e-9)).astype(np.uint8)
            clears.append((nm, load_rgb(im[..., :3])))
        except Exception: pass
    h = w = 384; yy, xx = np.mgrid[0:h, 0:w]                                            # procedural 'sky over field' scene
    sky = np.stack([0.62 + 0.3 * (1 - yy / h), 0.72 + 0.22 * (1 - yy / h), 0.9 + 0.08 * (1 - yy / h)], -1)
    field = np.stack([0.25 + 0.15 * rng.random((h, w)), 0.45 + 0.2 * rng.random((h, w)), 0.15 + 0.1 * rng.random((h, w))], -1)
    field = cv2.GaussianBlur(field, (0, 0), 1.2); scene = np.where((yy < h * 0.45)[..., None], sky, field)
    tree = ((xx - w * 0.7) ** 2 / 900 + (yy - h * 0.5) ** 2 / 3600) < 1; scene[tree] = [0.1, 0.25, 0.08]
    clears.append(("skyfield", np.clip(scene, 0, 1)))
    recs = []
    for nm, c in clears:
        for beta in (0.8, 1.6, 2.6):
            for A in ([0.85, 0.87, 0.90], [0.75, 0.77, 0.80]):
                kind = ["vertical", "radial", "smooth"][len(recs) % 3]
                depth = _depth_map(c.shape[0], c.shape[1], kind, rng)
                hz, _ = synth_haze(c, beta, A, depth)
                recs.append(dict(dataset="synthetic", real=False, train=False, key=f"synthetic:{nm}", name=f"{nm}_b{beta}_A{A[0]}",
                                 hazy=hz.astype(np.float32), gt=c.astype(np.float32), beta=beta, A_true=A))
    return recs

def build_records():
    """Evaluation records: every discovered benchmark pair except training sets, capped per dataset (seeded)."""
    allrecs = discover_pairs(INPUT_ROOT)
    tr = sorted({r["dataset"] for r in allrecs if r.get("train")})
    if tr: print("training-only sets found (excluded from evaluation, available to NB 3):", {t: sum(r["dataset"] == t for r in allrecs) for t in tr})
    recs = [r for r in allrecs if not r.get("train")]
    if recs:
        rng = np.random.default_rng(SEED); out = []
        for ds in sorted({r["dataset"] for r in recs}):
            sub = [r for r in recs if r["dataset"] == ds]
            if len(sub) > CONFIG["max_per_dataset"]:
                sub = [sub[i] for i in sorted(rng.choice(len(sub), CONFIG["max_per_dataset"], replace=False))]
            out += sub
        source = "benchmarks under " + str(INPUT_ROOT)
        return out, source
    return synthetic_records(), "synthetic-haze fallback (no paired dataset attached)"

def split_tune_test(recs, frac=None, seed=SEED):
    """Scene-grouped split: every hazy version of one clear scene lands in the same split (SOTS has 10 hazy/scene)."""
    frac = frac or CONFIG["tune_frac"]; rng = np.random.default_rng(seed)
    for ds in sorted({r["dataset"] for r in recs}):
        keys = sorted({r["key"] for r in recs if r["dataset"] == ds}); rng.shuffle(keys)
        n_t = min(len(keys) // 2, max(CONFIG["min_tune"], int(round(frac * len(keys))))) if len(keys) > 1 else 0
        tune = set(keys[:n_t])
        for r in recs:
            if r["dataset"] == ds: r["split"] = "tune" if r["key"] in tune else "test"
    return recs

def get_pair(rec):
    return (rec["hazy"].astype(np.float64), rec["gt"].astype(np.float64)) if isinstance(rec["hazy"], np.ndarray) else load_pair(rec)

def fingerprint(recs):
    import hashlib
    return hashlib.sha256("|".join(sorted(f"{r['dataset']}/{r['name']}/{r['split']}" for r in recs)).encode()).hexdigest()[:12]

def tex_macros(d, path):
    """Write \\newcommand macros (letters only) so the LaTeX report pulls every number from the run."""
    def clean(k): return re.sub(r"[^A-Za-z]", "", k)
    with open(path, "w", encoding="utf-8") as f:
        for k, v in d.items():
            f.write(f"\\newcommand{{\\{clean(k)}}}{{{v}}}\n")


## 1. Data, split, tuned configuration (must match NB 2's fingerprint)


In [ ]:
# ===== paired data: benchmarks attached under /kaggle/input, else the synthetic fallback =====
records, SOURCE = build_records()
records = split_tune_test(records)
FP = fingerprint(records)
inv = (pd.DataFrame([{"dataset": r["dataset"], "real": r["real"], "split": r["split"]} for r in records])
         .groupby(["dataset", "real", "split"]).size().unstack("split").fillna(0).astype(int))
inv["total"] = inv.sum(1)
print(f"source: {SOURCE}\nimages: {len(records)} | split fingerprint {FP} (scene-grouped tune/test, seed {SEED})")
print(inv.to_string())

stat_rows = []
for r in records:
    hz, g = get_pair(r); m = bright_mask(g)
    stat_rows.append({"name": r["name"], "density": haze_density(hz), "bright_px": int(m.sum()), "bright_frac": float(m.mean())})
S = pd.DataFrame(stat_rows); q1, q2 = S.density.quantile([1/3, 2/3]).values
S["stratum"] = np.where(S.density <= q1, "light", np.where(S.density <= q2, "medium", "dense")); STRATA = ["light", "medium", "dense"]
for r, st in zip(records, S.stratum): r["stratum"] = st
BASE = dict(patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3)
cands = list(INPUT_ROOT.rglob("dehaze_results.json")) + list(WORK.glob("dehaze_results.json"))
if cands:
    prev = json.load(open(cands[0])); TUNED = prev["tuned_cfg"]
    assert prev.get("fingerprint") == FP, f"NB 2 ran on a different split ({prev.get('fingerprint')} vs {FP}) — attach the matching NB 2 output"
    print("tuned DCP config from", cands[0], "->", TUNED)
else:
    TUNED = dict(BASE); print("WARNING: NB 2 output not attached -> scaffold = He et al. baseline. Attach NB 2's output for the paper run.")
if TUNED["refine"] == "none": TUNED = {**TUNED, "refine": "guided"}          # the learned refiner corrects a guided estimate
test_recs = [r for r in records if r["split"] == "test"]
print(f"test images: {len(test_recs)} | strata thresholds {q1:.3f} / {q2:.3f}")


## 2. Training data — RESIDE-ITS/OTS if attached (training-only, never evaluated), else synthetic haze from tune-split ground truths
The unsupervised arm additionally sees the **real hazy images of the tune split** (no ground truth is used).


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as Fnn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [0] if (FAST or device.type != "cuda") else [0, 1, 2]
STEPS = 40 if FAST else (3000 if device.type == "cuda" else 600)
BATCH, CROP, TRAIN_SIDE, N_TRAIN_CACHE = 8, 224, 320, 1500
print(f"torch {torch.__version__} | device {device} | seeds {SEEDS} | steps {STEPS}")

rng = np.random.default_rng(SEED)
train_pairs = [p for p in discover_pairs(INPUT_ROOT) if p.get("train")]
def _u8(a): return (np.clip(a, 0, 1) * 255).astype(np.uint8)
if len(train_pairs) >= 200:
    sel = [train_pairs[i] for i in rng.choice(len(train_pairs), min(N_TRAIN_CACHE, len(train_pairs)), replace=False)]
    TRAIN = [(_u8(load_rgb(p["hazy"], TRAIN_SIDE)), _u8(load_rgb(p["gt"], TRAIN_SIDE))) for p in sel]
    TRAIN = [(h, g if g.shape == h.shape else _u8(cv2.resize(g.astype(np.float64) / 255, (h.shape[1], h.shape[0])))) for h, g in TRAIN]
    TRAIN_SOURCE = f"{len(TRAIN)} RESIDE training pairs ({sel[0]['dataset']})"
else:
    tune_gts = [get_pair(r)[1] for r in records if r["split"] == "tune"]; TRAIN = None
    TRAIN_SOURCE = f"synthetic haze from {len(tune_gts)} tune-split ground truths (test scenes unseen)"
REAL_HAZY = [_u8(get_pair(r)[0]) for r in records if r["split"] == "tune" and r["real"]]
print("supervised training data :", TRAIN_SOURCE); print("extra unpaired real hazy  :", len(REAL_HAZY), "tune-split images")

def _crop_pair(h, g, c):
    H, W = h.shape[:2]; y0, x0 = rng.integers(0, H - c + 1), rng.integers(0, W - c + 1)
    return h[y0:y0+c, x0:x0+c], g[y0:y0+c, x0:x0+c]
def batch_pairs(bs=BATCH, crop=CROP):
    """(hazy, clear) float tensors (B,3,c,c) in [0,1]; one crop size per batch."""
    if TRAIN is not None:
        items = [TRAIN[i] for i in rng.choice(len(TRAIN), bs)]
        c = min([crop] + [min(h.shape[:2]) for h, _ in items]); xs, ys = zip(*[_crop_pair(h, g, c) for h, g in items])
        x = np.stack(xs).astype(np.float32) / 255; y = np.stack(ys).astype(np.float32) / 255
    else:
        gts = [tune_gts[i] for i in rng.choice(len(tune_gts), bs)]; c = min([crop] + [min(g.shape[:2]) for g in gts]); xs, ys = [], []
        for g in gts:
            _, gc = _crop_pair(g, g, c); depth = _depth_map(c, c, ["vertical", "radial", "smooth"][rng.integers(3)], rng)
            a = float(rng.uniform(0.7, 1.0)); hz, _ = synth_haze(gc, float(rng.uniform(0.5, 3.0)), np.clip([a, a + rng.uniform(-.03, .03), a + rng.uniform(-.03, .03)], 0, 1), depth)
            xs.append(hz); ys.append(gc)
        x = np.stack(xs).astype(np.float32); y = np.stack(ys).astype(np.float32)
    return torch.from_numpy(x).permute(0, 3, 1, 2).to(device), torch.from_numpy(y).permute(0, 3, 1, 2).to(device)
def batch_hazy_only(bs=BATCH, crop=CROP):
    """Hazy crops only (for the prior-as-loss arm): training hazies + real tune-split hazies, no ground truth."""
    if REAL_HAZY and (TRAIN is None or rng.random() < 0.3):
        items = [REAL_HAZY[i] for i in rng.choice(len(REAL_HAZY), bs)]
        c = min([crop] + [min(h.shape[:2]) for h in items]); x = np.stack([_crop_pair(h, h, c)[0] for h in items]).astype(np.float32) / 255
        return torch.from_numpy(x).permute(0, 3, 1, 2).to(device)
    return batch_pairs(bs, crop)[0]
x_, y_ = batch_pairs(); print("batch:", tuple(x_.shape), "->", tuple(y_.shape))


## 3. Differentiable classical stages (training) + the three residual modules
Erosion = min-pooling, guided filter = box filters, recovery = the closed-form equation, so gradients flow through the fixed stages. At evaluation the **NumPy** stages of NB 1–2 are used and only the learned module runs in PyTorch.


In [ ]:
def T_dark(x, k):                                            # (B,3,H,W) -> (B,1,H,W): channel-min then k×k min-pool (grey erosion)
    m = x.min(1, keepdim=True).values; p = int(k) // 2
    return -Fnn.max_pool2d(-Fnn.pad(m, (p, p, p, p), mode="replicate"), int(k), stride=1)
def T_box(x, r):
    r = int(r); return Fnn.avg_pool2d(Fnn.pad(x, (r, r, r, r), mode="replicate"), 2 * r + 1, stride=1)
def T_gray(x): return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
def T_guided(g, s, r, eps):
    mI, mp = T_box(g, r), T_box(s, r); cov = T_box(g * s, r) - mI * mp
    var = T_box(g * g, r) - mI * mI; a = cov / (var + eps); b = mp - a * mI
    return T_box(a, r) * g + T_box(b, r)
def T_transmission(x, A, omega, k): return 1.0 - omega * T_dark(x / A.view(-1, 3, 1, 1).clamp_min(1e-6), k)
def T_recover(x, A, t, t0): A = A.view(-1, 3, 1, 1); return (x - A) / t.clamp(t0, 1.0) + A       # unclamped on purpose
def A_classical(x, dark, method):
    """Classical A per sample via the NumPy estimator of NB 1 (identical train/eval)."""
    xn, dn = x.detach().permute(0, 2, 3, 1).cpu().numpy().astype(np.float64), dark.detach()[:, 0].cpu().numpy().astype(np.float64)
    return torch.tensor(np.stack([estimate_A(xn[i], dn[i], method) for i in range(len(xn))]), dtype=torch.float32, device=x.device)
def classical_intermediates(x, cfg):
    """dark, A_cls, t_raw, t_guided for a batch, all with the tuned configuration."""
    dark = T_dark(x, cfg["patch"]); A = A_classical(x, dark, cfg["A_method"])
    t_raw = T_transmission(x, A, cfg["omega"], cfg["patch"]); t_g = T_guided(T_gray(x), t_raw, cfg["gf_r"], cfg["gf_eps"]).clamp(0, 1)
    return dark, A, t_raw, t_g

def conv_block(i, o, d=1): return nn.Sequential(nn.Conv2d(i, o, 3, padding=d, dilation=d), nn.ReLU(inplace=True))
class TNet(nn.Module):
    """Transmission refiner: [hazy(3), t_raw, t_guided, dark] -> Δt (bounded), t = t_guided + Δt."""
    def __init__(self, ch=24):
        super().__init__(); self.body = nn.Sequential(conv_block(6, ch), conv_block(ch, ch, 2), conv_block(ch, ch, 4), conv_block(ch, ch, 8), conv_block(ch, ch), nn.Conv2d(ch, 1, 3, padding=1))
    def forward(self, x, t_raw, t_g, dark): return (t_g + 0.5 * torch.tanh(self.body(torch.cat([x, t_raw, t_g, dark], 1)))).clamp(0, 1)
class ANet(nn.Module):
    """Atmospheric-light corrector: hazy (downsampled) + broadcast classical A -> ΔA (bounded), A = A_cls + ΔA."""
    def __init__(self, ch=16):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(6, ch, 3, stride=2, padding=1), nn.ReLU(True), nn.Conv2d(ch, ch, 3, stride=2, padding=1), nn.ReLU(True),
                                  nn.Conv2d(ch, 2 * ch, 3, stride=2, padding=1), nn.ReLU(True), nn.Conv2d(2 * ch, 2 * ch, 3, stride=2, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(2 * ch, 3)
    def forward(self, x, A_cls):
        xs = Fnn.interpolate(x, size=(128, 128), mode="area"); z = self.body(torch.cat([xs, A_cls.view(-1, 3, 1, 1).expand(-1, 3, 128, 128)], 1)).flatten(1)
        return (A_cls + 0.3 * torch.tanh(self.fc(z))).clamp(0.05, 1.0)
class JNet(nn.Module):
    """Recovery corrector: [hazy(3), J_dcp(3), t] -> ΔJ (bounded), J = J_dcp + ΔJ."""
    def __init__(self, ch=24):
        super().__init__(); self.body = nn.Sequential(conv_block(7, ch), conv_block(ch, ch, 2), conv_block(ch, ch, 4), conv_block(ch, ch), nn.Conv2d(ch, 3, 3, padding=1))
    def forward(self, x, J, t): return (J + 0.25 * torch.tanh(self.body(torch.cat([x, J, t], 1)))).clamp(0, 1)
class AODNet(nn.Module):
    """Li et al. 2017 (as in NB 3) — the all-learned end point, trained here with the same data and budget."""
    def __init__(self, b=1.0):
        super().__init__(); self.b = b
        self.e_conv1 = nn.Conv2d(3, 3, 1); self.e_conv2 = nn.Conv2d(3, 3, 3, padding=1); self.e_conv3 = nn.Conv2d(6, 3, 5, padding=2); self.e_conv4 = nn.Conv2d(6, 3, 7, padding=3); self.e_conv5 = nn.Conv2d(12, 3, 3, padding=1)
    def forward(self, x):
        x1 = Fnn.relu(self.e_conv1(x)); x2 = Fnn.relu(self.e_conv2(x1)); x3 = Fnn.relu(self.e_conv3(torch.cat([x1, x2], 1)))
        x4 = Fnn.relu(self.e_conv4(torch.cat([x2, x3], 1))); k = Fnn.relu(self.e_conv5(torch.cat([x1, x2, x3, x4], 1))); return Fnn.relu(k * x - k + self.b)
n_params = lambda m: sum(p.numel() for p in m.parameters())
print({n: n_params(m()) for n, m in [("TNet", TNet), ("ANet", ANet), ("JNet", JNet), ("AODNet", AODNet)]}, "parameters")


## 4. Training — identical budget for every learned arm
Losses: **H-A / H-t (sup) / H-J / AOD-Net** = L1 to the clear image (through the fixed classical stages). **H-t (prior)** = mean dark channel of the recovered image (the prior itself) + L1 anchor to the guided estimate + total variation + a range penalty, on hazy images only.


In [ ]:
def tv(t): return (t[..., 1:, :] - t[..., :-1, :]).abs().mean() + (t[..., :, 1:] - t[..., :, :-1]).abs().mean()
LAMBDA = dict(dcp=1.0, anchor=1.0, tv=0.1, rng=1.0)
def loss_for(arm, net, x, y):
    if arm == "AOD-Net": return (net(x) - y).abs().mean()
    dark, A, t_raw, t_g = classical_intermediates(x, TUNED)
    if arm == "H-A":
        A2 = net(x, A); t2 = T_guided(T_gray(x), T_transmission(x, A2, TUNED["omega"], TUNED["patch"]), TUNED["gf_r"], TUNED["gf_eps"]).clamp(0, 1)
        return (T_recover(x, A2, t2, TUNED["t0"]).clamp(0, 1) - y).abs().mean()
    if arm == "H-t (sup)":
        return (T_recover(x, A, net(x, t_raw, t_g, dark), TUNED["t0"]).clamp(0, 1) - y).abs().mean()
    if arm == "H-t (prior)":
        t = net(x, t_raw, t_g, dark); J = T_recover(x, A, t, TUNED["t0"])
        return LAMBDA["dcp"] * T_dark(J.clamp(0, 1), 15).mean() + LAMBDA["anchor"] * (t - t_g).abs().mean() + LAMBDA["tv"] * tv(t) + LAMBDA["rng"] * (Fnn.relu(-J).mean() + Fnn.relu(J - 1).mean())
    if arm == "H-J":
        J = T_recover(x, A, t_g, TUNED["t0"]).clamp(0, 1); return (net(x, J, t_g) - y).abs().mean()
ARMS_LEARNED = {"H-A": ANet, "H-t (sup)": TNet, "H-t (prior)": TNet, "H-J": JNet, "AOD-Net": AODNet}
MODELS, HIST, TRAIN_MIN = {}, {}, {}
for seed in SEEDS:
    for arm, cls in ARMS_LEARNED.items():
        torch.manual_seed(seed); rng = np.random.default_rng(SEED + seed); net = cls().to(device)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
        hist, t0_ = [], time.time(); net.train()
        for step in range(1, STEPS + 1):
            if arm == "H-t (prior)": x = batch_hazy_only(); y = None
            else: x, y = batch_pairs()
            loss = loss_for(arm, net, x, y); opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step(); sched.step()
            hist.append(loss.item())
        net.eval(); MODELS[(arm, seed)] = net; HIST[(arm, seed)] = hist; TRAIN_MIN[(arm, seed)] = (time.time() - t0_) / 60
        torch.save(net.state_dict(), WORK / f"hybrid_{arm.replace(' ', '_').replace('(', '').replace(')', '')}_seed{seed}.pth")
        print(f"  seed {seed} | {arm:12s} | final loss {np.mean(hist[-max(1, STEPS // 20):]):.4f} | {TRAIN_MIN[(arm, seed)]:.1f} min")
fig, ax = plt.subplots(1, len(ARMS_LEARNED), figsize=(3.0 * len(ARMS_LEARNED), 3.2))
for a, arm in zip(ax, ARMS_LEARNED):
    for seed in SEEDS:
        h = np.array(HIST[(arm, seed)]); w = max(1, len(h) // 50); sm = np.convolve(h, np.ones(w) / w, mode="valid")
        a.plot(np.arange(len(sm)) + w, sm, lw=1.4, color=C_BLUE if seed == 0 else (C_ORANGE if seed == 1 else C_AQUA), label=f"seed {seed}")
    a.set_title(arm); a.set_xlabel("step"); a.set_yscale("log")
ax[0].set_ylabel("training loss (smoothed)"); ax[0].legend(fontsize=7)
fig.suptitle(f"Training curves — {TRAIN_SOURCE}", y=1.02, fontweight="bold"); plt.tight_layout(); savefig("fig_hybrid_training"); plt.show()


## 5. Evaluation on the test split — every arm, every seed, NumPy classical stages + the learned module
Per-image scores are averaged over seeds before the paired statistics (seed spread is kept in the CSV). Bright-region errors use masks from the clear image.


In [ ]:
def to_t(a): return torch.tensor(np.asarray(a, np.float32)).permute(2, 0, 1)[None].to(device) if a.ndim == 3 else torch.tensor(np.asarray(a, np.float32))[None, None].to(device)
def to_n(t): return t[0].permute(1, 2, 0).detach().cpu().numpy().astype(np.float64) if t.shape[1] == 3 else t[0, 0].detach().cpu().numpy().astype(np.float64)
@torch.no_grad()
def run_arms(hz, seed):
    """Returns {arm: (output image, ms)} for one hazy image and one seed of the learned modules."""
    out = {}
    t0_ = time.time(); out["hazy input"] = (hz, 0.0)
    t0_ = time.time(); out["CLAHE"] = (clahe_enhance(hz), (time.time() - t0_) * 1000)
    t0_ = time.time(); out["DCP baseline"] = (dehaze(hz, **BASE)["dehazed"], (time.time() - t0_) * 1000)
    t0_ = time.time(); o = dehaze(hz, **TUNED); ms_cls = (time.time() - t0_) * 1000; out["DCP tuned"] = (o["dehazed"], ms_cls)
    x, dark, t_raw, t_g = to_t(hz), to_t(o["dark"]), to_t(o["t_raw"]), to_t(o["t"]); A_cls = torch.tensor(o["A"], dtype=torch.float32, device=device)[None]
    # H-A: learned A -> classical transmission / guided / recovery (NumPy)
    t0_ = time.time(); A2 = MODELS[("H-A", seed)](x, A_cls)[0].cpu().numpy().astype(np.float64)
    tr2 = transmission_raw(hz, A2, TUNED["omega"], TUNED["patch"]); tg2 = np.clip(guided_filter(to_gray(hz), tr2, TUNED["gf_r"], TUNED["gf_eps"]), 0, 1)
    out["H-A"] = (recover(hz, A2, tg2, TUNED["t0"]), (time.time() - t0_) * 1000 + ms_cls)
    # H-t (sup / prior): learned refinement -> classical recovery
    for arm in ("H-t (sup)", "H-t (prior)"):
        t0_ = time.time(); t2 = to_n(MODELS[(arm, seed)](x, t_raw, t_g, dark)); out[arm] = (recover(hz, o["A"], t2, TUNED["t0"]), (time.time() - t0_) * 1000 + ms_cls)
        if arm == "H-t (sup)": t_sup = t2
        else: t_prior = t2
    # H-J: classical output -> learned correction
    t0_ = time.time(); out["H-J"] = (to_n(MODELS[("H-J", seed)](x, to_t(o["dehazed"]), t_g)), (time.time() - t0_) * 1000 + ms_cls)
    # H-all: learned A -> classical t_raw/guided -> learned refinement -> classical recovery -> learned correction
    t0_ = time.time(); tg2t = to_t(tg2); t3 = MODELS[("H-t (sup)", seed)](x, to_t(tr2), tg2t, dark); J3 = recover(hz, A2, to_n(t3), TUNED["t0"])
    out["H-all"] = (to_n(MODELS[("H-J", seed)](x, to_t(J3), t3)), (time.time() - t0_) * 1000 + ms_cls)
    t0_ = time.time(); out["AOD-Net"] = (np.clip(to_n(MODELS[("AOD-Net", seed)](x)), 0, 1), (time.time() - t0_) * 1000)
    return out, dict(t_raw=o["t_raw"], t_guided=o["t"], t_sup=t_sup, t_prior=t_prior)
ARMS = ["hazy input", "CLAHE", "DCP baseline", "DCP tuned", "H-A", "H-t (sup)", "H-t (prior)", "H-J", "H-all", "AOD-Net"]
rows, TMAPS, t_start = [], {}, time.time()
for i, r in enumerate(test_recs):
    hz, g = get_pair(r); m = bright_mask(g); has_b = int(m.sum()) >= CONFIG["min_bright_px"]
    for seed in SEEDS:
        outs, tmaps = run_arms(hz, seed)
        if seed == SEEDS[0] and has_b and r["name"] not in TMAPS and len(TMAPS) < 3: TMAPS[r["name"]] = tmaps
        for arm in ARMS:
            if seed != SEEDS[0] and arm in ("hazy input", "CLAHE", "DCP baseline", "DCP tuned"): continue     # classical arms do not depend on the seed
            J, ms = outs[arm]; err = np.abs(J - g).mean(2)
            rows.append({"name": r["name"], "dataset": r["dataset"], "stratum": r["stratum"], "real": r["real"], "seed": seed, "arm": arm, **score_all(J, g, hz), "ms": ms,
                         "err_bright": float(err[m].mean()) if has_b else np.nan, "err_else": float(err[~m].mean()) if has_b else np.nan, "has_bright": has_b})
    if (i + 1) % 25 == 0 or i == len(test_recs) - 1:
        el = time.time() - t_start; print(f"  {i + 1}/{len(test_recs)} | {el / 60:.1f} min | ~{el / (i + 1) * (len(test_recs) - i - 1) / 60:.1f} min left")
E = pd.DataFrame(rows); E.to_csv(WORK / "dehaze_hybrid_per_image.csv", index=False)
Es = E.groupby(["name", "dataset", "stratum", "real", "arm"], as_index=False).mean(numeric_only=True)          # seed-averaged per image
print("per-image rows:", E.shape, "| seed-averaged:", Es.shape)


## 6. Learning gain per stage (H4) — paired against the tuned DCP


In [ ]:
ref = Es[Es.arm == "DCP tuned"].set_index("name")
def arm_table(df, arms):
    out = []
    for arm in arms:
        sub = df[df.arm == arm].set_index("name").loc[ref.index]; mp, lo, hi = mean_ci(sub.psnr); ps = paired_stats(sub.psnr, ref.psnr); ss = paired_stats(sub.ssim, ref.ssim)
        out.append({"arm": arm, "PSNR": mp, "PSNR lo": lo, "PSNR hi": hi, "SSIM": sub.ssim.mean(), "CIEDE": sub.ciede.mean(), "e": sub.e.mean(), "r": sub.r.mean(),
                    "ΔPSNR vs tuned": ps["mean"], "Δ lo": ps["lo"], "Δ hi": ps["hi"], "p": ps["p"], "ΔSSIM": ss["mean"], "ms/img": sub.ms.mean()})
    return pd.DataFrame(out).set_index("arm")
main = arm_table(Es, ARMS)
seed_sd = E.groupby(["arm", "seed"]).psnr.mean().unstack("seed").std(1) if len(SEEDS) > 1 else pd.Series(0.0, index=ARMS)
main["seed sd"] = seed_sd.reindex(main.index).fillna(0.0)
table_png(main, "tab_hybrid_main", f"Every arm on the test split (n = {ref.shape[0]}, seeds = {len(SEEDS)}); paired Δ vs the tuned DCP", highlight_max=("PSNR", "SSIM"), highlight_min=("CIEDE",))
learned = ["H-A", "H-t (sup)", "H-t (prior)", "H-J", "H-all", "AOD-Net"]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(learned)); d = main.loc[learned]
cols = [C_VIOLET if a == "H-all" else (C_ORANGE if a == "AOD-Net" else (C_MAGENTA if a == "H-t (prior)" else C_BLUE)) for a in learned]
ax.bar(x, d["ΔPSNR vs tuned"], width=.62, color=cols, edgecolor=SURF)
ax.errorbar(x, d["ΔPSNR vs tuned"], yerr=[d["ΔPSNR vs tuned"] - d["Δ lo"], d["Δ hi"] - d["ΔPSNR vs tuned"]], fmt="none", ecolor=INK2, capsize=3)
for xi, v, h_ in zip(x, d["ΔPSNR vs tuned"], d["Δ hi"]): ax.text(xi, max(v, h_) + 0.04, f"{v:+.2f}", ha="center", va="bottom", fontsize=8.5, color=INK)
ax.axhline(0, color=RULE, lw=1); ax.set_xticks(x); ax.set_xticklabels(learned, rotation=10); ax.set_ylabel("ΔPSNR vs tuned DCP (dB), paired 95% CI")
ax.set_title("Learning gain per replaced stage (H4)"); plt.tight_layout(); savefig("fig_hybrid_gain_per_stage"); plt.show()
best_stage = d.loc[[a for a in learned if a not in ("H-all", "AOD-Net")], "ΔPSNR vs tuned"].idxmax()
print("largest single-stage gain:", best_stage, f"{d.loc[best_stage, 'ΔPSNR vs tuned']:+.2f} dB")


## 7. Density strata and sim-to-real transfer (H6)


In [ ]:
def delta_by(group_col, order):
    out = {}
    for arm in learned:
        sub = Es[Es.arm == arm].set_index("name").loc[ref.index]; dd = (sub.psnr - ref.psnr).rename("d").reset_index().merge(Es[Es.arm == arm][["name", group_col]], on="name")
        out[arm] = {g: mean_ci(dd[dd[group_col] == g].d) for g in order}
    return out
by_st = delta_by("stratum", STRATA)
st_tab = pd.DataFrame({arm: {s: by_st[arm][s][0] for s in STRATA} for arm in learned}).T[STRATA]
table_png(st_tab, "tab_hybrid_by_stratum", "ΔPSNR vs tuned DCP by density stratum (dB)")
ds_list = sorted(Es.dataset.unique()); by_ds = delta_by("dataset", ds_list)
ds_tab = pd.DataFrame({arm: {dsn: by_ds[arm][dsn][0] for dsn in ds_list} for arm in learned}).T[ds_list]
table_png(ds_tab, "tab_hybrid_by_dataset", "ΔPSNR vs tuned DCP by dataset (dB)")
real_flag = Es.groupby("name").real.first(); has_both = real_flag.any() and (~real_flag).any()
sr_rows = []
for arm in learned:
    sub = Es[Es.arm == arm].set_index("name").loc[ref.index]; dd = (sub.psnr - ref.psnr)
    ms_, ml_, mh_ = mean_ci(dd[~real_flag.loc[dd.index]]) if (~real_flag).any() else (np.nan,) * 3
    mr_, rl_, rh_ = mean_ci(dd[real_flag.loc[dd.index]]) if real_flag.any() else (np.nan,) * 3
    sr_rows.append({"arm": arm, "gain synthetic": ms_, "syn lo": ml_, "syn hi": mh_, "gain real": mr_, "real lo": rl_, "real hi": rh_, "sim-to-real drop": ms_ - mr_})
SR = pd.DataFrame(sr_rows).set_index("arm"); table_png(SR, "tab_hybrid_simreal", "Gain over tuned DCP on synthetic vs real haze (H6): drop = synthetic − real" + ("" if has_both else "  [only one kind of data attached]"))
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
w = 0.8 / len(learned)
for k, arm in enumerate(learned):
    vals = [by_st[arm][s][0] for s in STRATA]; los = [by_st[arm][s][1] for s in STRATA]; his = [by_st[arm][s][2] for s in STRATA]
    ax[0].bar(np.arange(3) + (k - len(learned) / 2 + .5) * w, vals, w, color=[C_BLUE, C_AQUA, C_MAGENTA, C_YELLOW, C_VIOLET, C_ORANGE][k], edgecolor=SURF, label=arm)
    ax[0].errorbar(np.arange(3) + (k - len(learned) / 2 + .5) * w, vals, yerr=[np.array(vals) - np.array(los), np.array(his) - np.array(vals)], fmt="none", ecolor=INK2, capsize=2, lw=.8)
ax[0].axhline(0, color=RULE, lw=1); ax[0].set_xticks(range(3)); ax[0].set_xticklabels(STRATA); ax[0].set_ylabel("ΔPSNR vs tuned DCP (dB)"); ax[0].set_title("Learning gain by density stratum"); ax[0].legend(fontsize=7, ncol=2)
if has_both:
    xx = np.arange(len(learned)); ax[1].bar(xx - .2, SR["gain synthetic"], .4, color=C_BLUE, edgecolor=SURF, label="synthetic sets"); ax[1].bar(xx + .2, SR["gain real"], .4, color=C_ORANGE, edgecolor=SURF, label="real sets")
    ax[1].errorbar(xx - .2, SR["gain synthetic"], yerr=[SR["gain synthetic"] - SR["syn lo"], SR["syn hi"] - SR["gain synthetic"]], fmt="none", ecolor=INK2, capsize=2, lw=.8)
    ax[1].errorbar(xx + .2, SR["gain real"], yerr=[SR["gain real"] - SR["real lo"], SR["real hi"] - SR["gain real"]], fmt="none", ecolor=INK2, capsize=2, lw=.8)
    ax[1].axhline(0, color=RULE, lw=1); ax[1].set_xticks(xx); ax[1].set_xticklabels(learned, rotation=10); ax[1].set_title("Sim-to-real transfer (H6)"); ax[1].legend(fontsize=8)
else:
    ax[1].axis("off"); ax[1].text(.5, .5, "attach both synthetic (SOTS) and real (O/I/Dense-HAZE)\nsets for the sim-to-real test", ha="center", va="center", color=INK2)
plt.tight_layout(); savefig("fig_hybrid_by_density"); plt.show()


## 8. Bright-region test (H5) — does a learned stage beat its teacher where the prior is violated?
Masks come from the *clear* image. The key contrast is **H-t (prior)** vs **H-t (sup)**: same network, same inputs, different training signal.


In [ ]:
B = Es[Es.has_bright > 0.5]
if len(B):
    refb = B[B.arm == "DCP tuned"].set_index("name"); brows = []
    for arm in ARMS:
        sub = B[B.arm == arm].set_index("name").loc[refb.index]; pb = paired_stats(sub.err_bright, refb.err_bright); pe = paired_stats(sub.err_else, refb.err_else)
        brows.append({"arm": arm, "n": len(sub), "err bright": sub.err_bright.mean(), "err elsewhere": sub.err_else.mean(), "ratio": sub.err_bright.mean() / max(sub.err_else.mean(), 1e-9),
                      "Δ bright vs tuned": pb["mean"], "Δb lo": pb["lo"], "Δb hi": pb["hi"], "p bright": pb["p"], "Δ elsewhere vs tuned": pe["mean"], "p elsewhere": pe["p"]})
    BT = pd.DataFrame(brows).set_index("arm")
    table_png(BT, "tab_hybrid_bright", f"Error inside bright/sky regions vs elsewhere ({len(refb)} test images with a bright region); Δ < 0 = better than the tuned DCP", highlight_min=("err bright", "ratio"), fmt="{:.4f}")
    fig, ax = plt.subplots(figsize=(9.5, 4)); x = np.arange(len(ARMS)); w = .38
    ax.bar(x - w/2, BT["err bright"], w, color=C_RED, edgecolor=SURF, label="inside bright / sky mask"); ax.bar(x + w/2, BT["err elsewhere"], w, color=C_BLUE, edgecolor=SURF, label="elsewhere")
    for xi, (a_, b_) in enumerate(zip(BT["err bright"], BT["err elsewhere"])):
        ax.text(xi - w/2, a_, f"{a_:.3f}", ha="center", va="bottom", fontsize=7, color=INK); ax.text(xi + w/2, b_, f"{b_:.3f}", ha="center", va="bottom", fontsize=7, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(ARMS, rotation=15); ax.set_ylabel("mean |J − GT| per pixel"); ax.set_title("Bright-region failure per arm (H5)"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig("fig_hybrid_bright"); plt.show()
    BRIGHT = {arm: {k: float(BT.loc[arm, k]) for k in ("err bright", "err elsewhere", "ratio", "Δ bright vs tuned", "Δb lo", "Δb hi", "p bright")} for arm in ARMS}; BRIGHT["n"] = int(len(refb))
else:
    print("no test image has a bright region — H5 skipped"); BRIGHT = {"n": 0}
# transmission maps on a bright-region image: raw / guided / learned (sup) / learned (prior)
if TMAPS:
    nm, tm = next(iter(TMAPS.items())); r = {r_["name"]: r_ for r_ in records}[nm]; hz, g = get_pair(r)
    fig, ax = plt.subplots(2, 4, figsize=(13, 6.4))
    for j, (k, t) in enumerate([("raw", tm["t_raw"]), ("guided filter", tm["t_guided"]), ("learned (sup)", tm["t_sup"]), ("learned (prior loss)", tm["t_prior"])]):
        ax[0, j].imshow(t, cmap="gray", vmin=0, vmax=1); ax[0, j].set_title(f"t — {k}", fontsize=9); ax[0, j].axis("off")
        J = recover(hz, dehaze(hz, **TUNED)["A"], np.clip(t, 0, 1), TUNED["t0"]); ax[1, j].imshow(J, vmin=0, vmax=1); ax[1, j].set_title(f"J — {k}   {psnr(J, g):.1f} dB", fontsize=9); ax[1, j].axis("off")
    ax[1, 0].text(0.01, 0.98, f"{r['dataset']} · bright {bright_mask(g).mean():.0%}", transform=ax[1, 0].transAxes, fontsize=7.5, va="top", color="w", bbox=dict(facecolor=INK, alpha=.55, pad=2, lw=0))
    fig.suptitle("Transmission refinement on a bright-region test image: classical vs learned (sup) vs learned (prior loss)", y=1.0, fontweight="bold"); plt.tight_layout(); savefig("fig_hybrid_tmaps"); plt.show()


## 9. Cost, qualitative grid, results + LaTeX macros


In [ ]:
cost = pd.DataFrame({"params": {**{a: 0 for a in ARMS}, "H-A": n_params(ANet()), "H-t (sup)": n_params(TNet()), "H-t (prior)": n_params(TNet()), "H-J": n_params(JNet()), "H-all": n_params(ANet()) + n_params(TNet()) + n_params(JNet()), "AOD-Net": n_params(AODNet())},
                     "ms/img": main["ms/img"], "train min (per seed)": {**{a: 0.0 for a in ARMS}, **{a: float(np.mean([TRAIN_MIN[(a, s)] for s in SEEDS])) for a in ARMS_LEARNED}, "H-all": float(np.mean([TRAIN_MIN[(a, s)] for a in ("H-A", "H-t (sup)", "H-J") for s in SEEDS])) * 3}}).loc[ARMS]
table_png(cost, "tab_hybrid_cost", f"Cost per arm (learned modules on {device.type}; classical stages on CPU)", fmt="{:.1f}")
by_name = {r["name"]: r for r in records}; order = ref.psnr.sort_values().index
picks = [order[0], order[len(order) // 3], order[2 * len(order) // 3], order[-1]]; show = ["hazy input", "DCP tuned", "H-A", "H-t (sup)", "H-t (prior)", "H-J", "H-all", "AOD-Net"]
fig, ax = plt.subplots(len(picks), len(show) + 1, figsize=(2.2 * (len(show) + 1), 2.3 * len(picks)))
for i, nm in enumerate(picks):
    r = by_name[nm]; hz, g = get_pair(r); outs, _ = run_arms(hz, SEEDS[0])
    for j, arm in enumerate(show):
        ax[i, j].imshow(outs[arm][0], vmin=0, vmax=1); ax[i, j].set_title(f"{arm}  {psnr(outs[arm][0], g):.1f}", fontsize=7); ax[i, j].axis("off")
    ax[i, -1].imshow(g, vmin=0, vmax=1); ax[i, -1].set_title("ground truth", fontsize=7); ax[i, -1].axis("off")
    ax[i, 0].text(0.01, 0.98, f"{r['dataset']} · {r['stratum']}", transform=ax[i, 0].transAxes, fontsize=6.5, va="top", color="w", bbox=dict(facecolor=INK, alpha=.55, pad=2, lw=0))
fig.suptitle("Same test images, every arm (rows: worst → best for the tuned DCP)", y=1.0, fontweight="bold"); plt.tight_layout(); savefig("fig_hybrid_qualitative"); plt.show()

res = {"source": SOURCE, "fingerprint": FP, "train_source": TRAIN_SOURCE, "n_real_hazy_unpaired": len(REAL_HAZY), "device": device.type, "seeds": SEEDS, "steps": STEPS, "tuned_cfg": TUNED, "lambda": LAMBDA,
       "main": main.round(4).to_dict("index"), "by_stratum": st_tab.round(3).to_dict("index"), "by_dataset": ds_tab.round(3).to_dict("index"), "simreal": SR.round(3).to_dict("index"),
       "bright": BRIGHT, "cost": cost.round(3).to_dict("index"), "best_single_stage": best_stage}
json.dump(res, open(WORK / "dehaze_hybrid_results.json", "w"), indent=1, default=float)
fmt2 = lambda v: f"{v:.2f}"; fmt3 = lambda v: f"{v:.3f}"
def fmtp(p):
    """p-value for LaTeX math mode: 0.023, or 3.8 x 10^-5 (TeX times) below 1e-3."""
    if p != p: return r"\mathrm{n/a}"
    if p >= 1e-3: return f"{p:.3f}"
    m, e = f"{p:.1e}".split("e"); return m + "\\times10^{" + str(int(e)) + "}"
KEY = {"hazy input": "Input", "CLAHE": "Clahe", "DCP baseline": "Base", "DCP tuned": "Tuned", "H-A": "HA", "H-t (sup)": "HTsup", "H-t (prior)": "HTprior", "H-J": "HJ", "H-all": "HAll", "AOD-Net": "Aod"}
macros = {"hybNTest": int(ref.shape[0]), "hybSeeds": len(SEEDS), "hybSteps": STEPS, "hybTrainSource": TRAIN_SOURCE.replace("_", r"\_"), "hybNRealHazy": len(REAL_HAZY), "hybDevice": device.type,
          "hybBestStage": best_stage, "hybBestStageGain": fmt2(main.loc[best_stage, "ΔPSNR vs tuned"]), "hybBrightN": BRIGHT.get("n", 0)}
for arm in ARMS:
    k = KEY[arm]
    macros.update({f"hyb{k}PSNR": fmt2(main.loc[arm, "PSNR"]), f"hyb{k}SSIM": fmt3(main.loc[arm, "SSIM"]), f"hyb{k}CIEDE": fmt2(main.loc[arm, "CIEDE"]), f"hyb{k}Delta": fmt2(main.loc[arm, "ΔPSNR vs tuned"]),
                   f"hyb{k}Lo": fmt2(main.loc[arm, "Δ lo"]), f"hyb{k}Hi": fmt2(main.loc[arm, "Δ hi"]), f"hyb{k}P": fmtp(main.loc[arm, "p"]), f"hyb{k}Ms": f"{main.loc[arm, 'ms/img']:.0f}", f"hyb{k}Params": f"{int(cost.loc[arm, 'params']):,}".replace(",", r"{,}")})
    if arm in learned:
        macros.update({f"hyb{k}Light": fmt2(st_tab.loc[arm, "light"]), f"hyb{k}Medium": fmt2(st_tab.loc[arm, "medium"]), f"hyb{k}Dense": fmt2(st_tab.loc[arm, "dense"]),
                       f"hyb{k}Synth": fmt2(SR.loc[arm, "gain synthetic"]), f"hyb{k}Real": fmt2(SR.loc[arm, "gain real"]), f"hyb{k}Drop": fmt2(SR.loc[arm, "sim-to-real drop"])})
    if BRIGHT.get("n", 0):
        macros.update({f"hyb{k}Bright": fmt3(BRIGHT[arm]["err bright"]), f"hyb{k}Else": fmt3(BRIGHT[arm]["err elsewhere"]), f"hyb{k}Ratio": fmt2(BRIGHT[arm]["ratio"]),
                       f"hyb{k}BDelta": f"{BRIGHT[arm]['Δ bright vs tuned']:+.4f}", f"hyb{k}BLo": f"{BRIGHT[arm]['Δb lo']:+.4f}", f"hyb{k}BHi": f"{BRIGHT[arm]['Δb hi']:+.4f}", f"hyb{k}BP": fmtp(BRIGHT[arm]["p bright"])})
tex_macros(macros, WORK / "numbers_hybrid.tex")
print(json.dumps({"train_source": TRAIN_SOURCE, "best_single_stage": best_stage, "delta_vs_tuned": {a: round(main.loc[a, "ΔPSNR vs tuned"], 3) for a in learned}}, indent=1))


## Summary
* **H4** — `tab_hybrid_main` / `fig_hybrid_gain_per_stage`: the paired gain of replacing each stage, with CIs, seeds and cost.
* **H5** — `tab_hybrid_bright` / `fig_hybrid_tmaps`: the same refiner trained with the prior as its loss vs with ground truth, inside bright regions.
* **H6** — `tab_hybrid_simreal` / `fig_hybrid_by_density`: gain on synthetic vs real haze per arm; gain by density stratum.
* `numbers_hybrid.tex` fills `report/report_hybrid.tex`.
